STEP 1 :- Import Libraries And Load Dataset.

In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import KNNImputer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest

# Load the messy dataset
df = pd.read_csv('real_estate_raw_messy.csv')

print("Original dataset shape:", df.shape)
display(df.head())

Original dataset shape: (12180, 26)


,Customer_ID,Age,Gender,Occupation,City,Monthly_Income,Annual_Income,Family_Size,Current_Housing_Status,Plot_Budget,...,Lead_Source,Enquiry_Date,Previous_Enquiry,Site_Visit,Negotiation_Done,Booking_Done,Purchase_Probability,Purchase_Intent,Purchased,Purchase_Value
0,CUST_20988,21,Male,Unknown,Mumbai,131500,1578000,2,Owned Flat/House,7020000,...,Google & Social Media Ads,2024-07-19,N,No,No,No,0.0639,Low,No,0
1,CUST_16662,29,Male,Real Estate & Construction,Kolkata,43800,525600,4,Owned Flat/House,"INR 2,000,000",...,Magicbricks / Real Estate Portal,01-13-2025,yes,No,No,No,0.0668,Low,No,0
2,CUST_10543,50,M,Retail & Commerce,Noida,1.79 Lakh/pm,2149200,2,Owned Flat/House,5710000,...,Channel Partner / Broker,25-Oct-2024,No,Y,Yes,Yes,0.9900,High,Yes,5550000
3,CUST_15741,44,Male,Retail & Commerce,Pune,45700,548400,4,Living with Parents,"2,300,000",...,Print / Newspaper Ad,2025-06-06,Yes,No,No,no,0.2293,Low,No,0
4,CUST_16911,38,Male,business owner,Ahmedabad,"₹219,400",2632800,4,Rented,Not Disclosed,...,Magicbricks / Real Estate Portal,Invalid Date,No,Yes,No,No,0.7678,High,Yes,8930000


STEP 2 :- Remove Duplicates & Clean Customer_ID.

In [2]:
initial_rows = len(df)

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

removed_duplicates = initial_rows - len(df)

# Remove leading/trailing whitespace
df['Customer_ID'] = (
    df['Customer_ID']
    .astype(str)
    .str.strip()
)

print(f"Removed {removed_duplicates} duplicate rows.")
print(f"Remaining rows: {len(df)}")

print("\nDuplicate Customer_IDs:")
print(df['Customer_ID'].duplicated().sum())

Removed 180 duplicate rows.
Remaining rows: 12000

Duplicate Customer_IDs:
0


STEP 3 :- Clean Age & Categorical Variables.

In [3]:
# 1. Clean Age

df['Age'] = (
    df['Age']
    .astype(str)
    .str.replace(' yrs', '', case=False)
    .str.strip()
)

df['Age'] = pd.to_numeric(
    df['Age'],
    errors='coerce'
)

df['Age'] = df['Age'].abs()

# Keep only realistic working age range
df.loc[
    (df['Age'] < 18) | (df['Age'] > 100),
    'Age'
] = np.nan


# 2. Clean Gender

gender_map = {
    'Male': 'Male',
    'M': 'Male',
    'male': 'Male',
    'MALE': 'Male',
    'Female': 'Female',
    'F': 'Female',
    'female': 'Female',
    'FEMALE': 'Female'
}

df['Gender'] = df['Gender'].map(gender_map)


# 3. Clean Occupation

occupation_map = {
    'IT & Software': 'IT & Software',
    'IT': 'IT & Software',
    'Software Engineer': 'IT & Software',
    'Tech / Software': 'IT & Software',
    'it / software': 'IT & Software',
    'IT Sector': 'IT & Software',

    'BFSI & Banking': 'BFSI & Banking',
    'Finance / BFSI': 'BFSI & Banking',
    'Banking': 'BFSI & Banking',
    'Banker': 'BFSI & Banking',
    'bfsi': 'BFSI & Banking',

    'Government & PSU': 'Government & PSU',
    'Government': 'Government & PSU',
    'govt job': 'Government & PSU',
    'Govt': 'Government & PSU',
    'PSU Employee': 'Government & PSU',

    'Self-Employed / Business': 'Self-Employed / Business',
    'business owner': 'Self-Employed / Business',
    'Self Employed': 'Self-Employed / Business',
    'Trader': 'Self-Employed / Business',
    'Business': 'Self-Employed / Business',

    'Healthcare & Doctor': 'Healthcare & Doctor',
    'Healthcare': 'Healthcare & Doctor',
    'Medical': 'Healthcare & Doctor',
    'Doctor': 'Healthcare & Doctor',

    'Real Estate & Construction': 'Real Estate & Construction',
    'Bbuilder / Contractor': 'Real Estate & Construction',
    'Realtor': 'Real Estate & Construction',
    'Real Estate': 'Real Estate & Construction',

    'Manufacturing & Engg': 'Manufacturing & Engg',
    'Retail & Commerce': 'Retail & Commerce',
    'Education & Academic': 'Education & Academic',
    'Retired': 'Retired'
}

df['Occupation'] = df['Occupation'].map(occupation_map)


# 4. Clean City

city_map = {
    'Mumbai': 'Mumbai',
    'MMR': 'Mumbai',
    'Bombay': 'Mumbai',
    'mumbai': 'Mumbai',
    'MUMBAI': 'Mumbai',

    'Bengaluru': 'Bengaluru',
    'Bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',
    'BANGALORE': 'Bengaluru',
    'BLR': 'Bengaluru',

    'Delhi': 'Delhi',
    'Delhi NCR': 'Delhi',
    'New Delhi': 'Delhi',
    'DELHI': 'Delhi',
    'delhi': 'Delhi',

    'Noida': 'Noida',
    'Greater Noida': 'Noida',
    'NOIDA': 'Noida',
    'noida': 'Noida',

    'Gurugram': 'Gurugram',
    'Gurgaon': 'Gurugram',
    'GURUGRAM': 'Gurugram',
    'gurgaon': 'Gurugram',

    'Hyderabad': 'Hyderabad',
    'Pune': 'Pune',
    'Chennai': 'Chennai',
    'Ahmedabad': 'Ahmedabad',
    'Kolkata': 'Kolkata'
}

df['City'] = df['City'].map(city_map)


# 5. Clean Preferred Location

location_map = {
    'Suburbs': 'Suburbs',
    'suburbs': 'Suburbs',

    'Periphery / Growth Corridor': 'Periphery / Growth Corridor',
    'periphery / growth corridor': 'Periphery / Growth Corridor',

    'Within City Limits': 'Within City Limits',
    'within city limits': 'Within City Limits',

    'City Centre': 'City Centre',
    'city centre': 'City Centre'
}

df['Preferred_Location'] = df[
    'Preferred_Location'
].map(location_map)


print("Age summary:")
print(df['Age'].describe())

print("\nMissing Age values:", df['Age'].isna().sum())
print("\nGender:")
print(df['Gender'].value_counts(dropna=False))

print("\nUnique Cities:", df['City'].nunique())
print("Unique Occupations:", df['Occupation'].nunique())
print("Unique Preferred Locations:", df['Preferred_Location'].nunique())

Age summary:
count    11678.000000
mean        38.459240
std         11.138951
min         18.000000
25%         30.000000
50%         37.000000
75%         44.000000
max         70.000000
Name: Age, dtype: float64

Missing Age values: 322

Gender:
Gender
Male      9130
Female    2506
NaN        364
Name: count, dtype: int64

Unique Cities: 10
Unique Occupations: 10
Unique Preferred Locations: 4


STEP 4 :- Clean Numerical Values.

In [4]:
# 1. Clean Plot Budget

df['Plot_Budget'] = (
    df['Plot_Budget']
    .astype(str)
    .str.replace('INR', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

df['Plot_Budget'] = df['Plot_Budget'].replace(
    ['Not Disclosed', '?', 'NaN', 'nan'],
    np.nan
)

mask_lakh_budget = df['Plot_Budget'].astype(str).str.contains(
    'Lakh',
    na=False
)

df['Plot_Budget'] = pd.to_numeric(
    df['Plot_Budget']
    .astype(str)
    .str.replace('Lakh', '', regex=False)
    .str.strip(),
    errors='coerce'
).abs()

df.loc[mask_lakh_budget, 'Plot_Budget'] *= 100000


# 2. Clean Monthly Income

df['Monthly_Income'] = (
    df['Monthly_Income']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

df['Monthly_Income'] = df['Monthly_Income'].replace(
    ['?', 'NaN', 'nan'],
    np.nan
)

mask_lakh_income = df['Monthly_Income'].astype(str).str.contains(
    'Lakh/pm',
    na=False
)

df['Monthly_Income'] = pd.to_numeric(
    df['Monthly_Income']
    .astype(str)
    .str.replace('Lakh/pm', '', regex=False)
    .str.strip(),
    errors='coerce'
).abs()

df.loc[mask_lakh_income, 'Monthly_Income'] *= 100000


# 3. Clean Preferred Plot Size

df['Preferred_Plot_Size_SqFt'] = (
    df['Preferred_Plot_Size_SqFt']
    .astype(str)
    .str.strip()
)

df['Preferred_Plot_Size_SqFt'] = df[
    'Preferred_Plot_Size_SqFt'
].replace(
    ['?', 'NaN', 'nan'],
    np.nan
)

mask_sqyd = df['Preferred_Plot_Size_SqFt'].astype(str).str.contains(
    'sq.yd',
    na=False
)

df['Preferred_Plot_Size_SqFt'] = pd.to_numeric(
    df['Preferred_Plot_Size_SqFt']
    .astype(str)
    .str.replace('sq.yd', '', regex=False)
    .str.replace('sq.ft', '', regex=False)
    .str.strip(),
    errors='coerce'
).abs()

# Convert square yards to square feet
df.loc[mask_sqyd, 'Preferred_Plot_Size_SqFt'] *= 9


# 4. Clean Distance

df['Distance_to_City_Center_km'] = (
    df['Distance_to_City_Center_km']
    .astype(str)
    .str.replace('km', '', case=False)
    .str.strip()
)

df['Distance_to_City_Center_km'] = pd.to_numeric(
    df['Distance_to_City_Center_km'],
    errors='coerce'
).abs()


# 5. Clean Dates

invalid_date_strings = [
    'Invalid Date',
    '?',
    'NaN',
    'null',
    'None'
]

invalid_dates_count = df['Enquiry_Date'].isin(
    invalid_date_strings
).sum()

df['Enquiry_Date'] = df['Enquiry_Date'].replace(
    invalid_date_strings,
    np.nan
)


print("Invalid dates converted to NaN:", invalid_dates_count)

print("\nMissing numerical values:")
print(
    df[
        [
            'Age',
            'Monthly_Income',
            'Plot_Budget',
            'Preferred_Plot_Size_SqFt',
            'Distance_to_City_Center_km'
        ]
    ].isna().sum()
)

Invalid dates converted to NaN: 106

Missing numerical values:
Age                            322
Monthly_Income                 704
Plot_Budget                    811
Preferred_Plot_Size_SqFt      1601
Distance_to_City_Center_km     630
dtype: int64


STEP 5 :- Binary Values & Sales Funnel Logic.

In [5]:
# 1. Standardize Binary Columns

binary_cols = [
    'Loan_Required',
    'Previous_Enquiry',
    'Site_Visit',
    'Negotiation_Done',
    'Booking_Done',
    'Purchased'
]

binary_map = {
    'Y': 1, 'y': 1, 'Yes': 1, 'yes': 1,
    'N': 0, 'n': 0, 'No': 0, 'no': 0,
    '1': 1, '0': 0, 1: 1, 0: 0
}

for col in binary_cols:
    df[col] = df[col].map(binary_map)

print("Binary columns standardized:")
for col in binary_cols:
    print(f"{col}: {df[col].value_counts(dropna=False).to_dict()}")



Binary columns standardized:
Loan_Required: {1.0: 9937, 0.0: 1704, nan: 359}
Previous_Enquiry: {0.0: 7167, 1.0: 4487, nan: 346}
Site_Visit: {0.0: 6237, 1.0: 5373, nan: 390}
Negotiation_Done: {0.0: 8807, 1.0: 2827, nan: 366}
Booking_Done: {0.0: 9776, 1.0: 1878, nan: 346}
Purchased: {0.0: 6811, 1.0: 4824, nan: 365}


STEP 6 :- Financial & Business Logic.

In [6]:
# 1. Enquiry implies Site Visit.
df.loc[
    df['Previous_Enquiry'] == 1,
    'Site_Visit'
] = 1


# 2. Purchase implies Booking.

df.loc[
    df['Purchased'] == 1,
    'Booking_Done'
] = 1


# 3. If Purchased = 0, Purchase_Value must be 0

df['Purchase_Value'] = pd.to_numeric(
    df['Purchase_Value'],
    errors='coerce'
)

df.loc[
    df['Purchased'] == 0,
    'Purchase_Value'
] = 0.0


# 4. Buyers with missing/zero Purchase_Value

df.loc[
    (df['Purchased'] == 1) &
    (
        df['Purchase_Value'].isna() |
        (df['Purchase_Value'] == 0)
    ),
    'Purchase_Value'
] = df['Plot_Budget']


print("Financial and business logic applied successfully.")

Financial and business logic applied successfully.


STEP 7 :- Missing Value Imputation.

In [7]:
# 1. Treat zero Monthly_Income as missing
# Zero income is considered invalid/unavailable.

df.loc[df['Monthly_Income'] <= 0, 'Monthly_Income'] = np.nan


# 2. Categorical Imputation

categorical_cols = [
    'Gender',
    'Occupation',
    'City',
    'Current_Housing_Status',
    'Preferred_Location',
    'Purpose',
    'Expected_Purchase_Timeline',
    'Lead_Source',
    'Purchase_Intent'
]

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


# 3. Numerical Imputation using MICE

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice_cols = [
    'Monthly_Income',
    'Age',
    'Family_Size',
    'Plot_Budget',
    'Preferred_Plot_Size_SqFt',
    'Distance_to_City_Center_km'
] + binary_cols

mice_imputer = IterativeImputer(
    max_iter=10,
    random_state=42
)

df[mice_cols] = mice_imputer.fit_transform(df[mice_cols])


# 4. Convert binary values back to 0/1

for col in binary_cols:
    df[col] = (
        df[col]
        .round()
        .clip(0, 1)
        .astype('int64')
    )


# 5. Ensure numerical columns are non-negative

non_negative_cols = [
    'Age',
    'Monthly_Income',
    'Family_Size',
    'Plot_Budget',
    'Preferred_Plot_Size_SqFt',
    'Distance_to_City_Center_km'
]

for col in non_negative_cols:
    df[col] = df[col].abs()


print("Remaining missing values:")
print(df.isna().sum().sum())

Remaining missing values:
316


STEP 8 :- Dates & Datatypes.

In [8]:
# 1. Parse Enquiry Date

df['Enquiry_Date'] = pd.to_datetime(
    df['Enquiry_Date'],
    format='mixed',
    errors='coerce'
)

missing_dates_before = df['Enquiry_Date'].isna().sum()

date_median = df['Enquiry_Date'].median()

df['Enquiry_Date'] = df[
    'Enquiry_Date'
].fillna(date_median)


# 2. Integer Columns

int_columns = [
    'Age',
    'Loan_Required',
    'Previous_Enquiry',
    'Site_Visit',
    'Negotiation_Done',
    'Booking_Done',
    'Purchased',
    'Family_Size'
]

for col in int_columns:
    df[col] = (
        df[col]
        .round()
        .astype('int64')
    )


# 3. Float Columns

float_columns = [
    'Monthly_Income',
    'Annual_Income',
    'Plot_Budget',
    'Preferred_Plot_Size_SqFt',
    'Distance_to_City_Center_km',
    'Purchase_Value',
    'Purchase_Probability'
]

for col in float_columns:
    df[col] = df[col].astype('float64')


# 4. String Columns

string_columns = [
    'Customer_ID',
    'Gender',
    'Occupation',
    'City',
    'Preferred_Location',
    'Current_Housing_Status',
    'Purpose',
    'Expected_Purchase_Timeline',
    'Lead_Source',
    'Purchase_Intent'
]

for col in string_columns:
    df[col] = df[col].astype(str)


# Recalculate derived financial field
df['Annual_Income'] = (
    df['Monthly_Income'] * 12
)


print(
    f"Parsed and imputed {missing_dates_before} "
    f"missing dates using median date "
    f"({date_median.strftime('%Y-%m-%d')})."
)

print(
    "\nTotal remaining nulls:",
    df.isnull().sum().sum()
)

Parsed and imputed 316 missing dates using median date (2024-09-30).

Total remaining nulls: 0


STEP 9 :- 99th Percentile Rule & Isolation Forest.

In [9]:
# 1. 99th Percentile Upper-Tail Treatment

percentile_cols = [
    'Age',
    'Monthly_Income',
    'Plot_Budget',
    'Preferred_Plot_Size_SqFt',
    'Distance_to_City_Center_km'
]

percentile_99_values = {}

print("99th Percentile Outlier Treatment")
print("=" * 65)

for col in percentile_cols:

    # Calculate the 99th percentile
    cap_value = df[col].quantile(0.99)

    percentile_99_values[col] = cap_value

    # Identify values above the 99th percentile
    outlier_mask = df[col] > cap_value

    outlier_count = outlier_mask.sum()

    # Cap extreme upper-tail values
    df.loc[outlier_mask, col] = cap_value

    print(
        f"{col:<35}"
        f"99th percentile = {cap_value:,.2f} | "
        f"Capped = {outlier_count}"
    )


# Annual income is derived from Monthly Income
df['Annual_Income'] = (
    df['Monthly_Income'] * 12
)


# 2. Multivariate Outlier Detection

num_features = [
    'Age',
    'Monthly_Income',
    'Annual_Income',
    'Plot_Budget',
    'Preferred_Plot_Size_SqFt',
    'Distance_to_City_Center_km'
]

iso_forest = IsolationForest(
    contamination=0.02,
    random_state=42
)

outlier_predictions = iso_forest.fit_predict(
    df[num_features]
)

df['Outlier_Flag'] = pd.Series(
    outlier_predictions,
    index=df.index
).map({
    1: 'Inlier',
    -1: 'Outlier'
})


# Verification

print("\nIsolation Forest Outlier Distribution:")
print(df['Outlier_Flag'].value_counts())

print("\nIsolation Forest Outlier Percentage:")
print(
    df['Outlier_Flag']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\n99th Percentile Verification:")

for col in percentile_cols:
    print(
        f"{col}: "
        f"Maximum = {df[col].max():,.2f}"
    )

print(
    "\nAnnual Income relationship:",
    np.isclose(
        df['Annual_Income'],
        df['Monthly_Income'] * 12
    ).all()
)

99th Percentile Outlier Treatment
Age                                99th percentile = 69.00 | Capped = 75
Monthly_Income                     99th percentile = 535,803.00 | Capped = 120
Plot_Budget                        99th percentile = 32,938,671.26 | Capped = 120
Preferred_Plot_Size_SqFt           99th percentile = 3,404.48 | Capped = 120
Distance_to_City_Center_km         99th percentile = 47.20 | Capped = 107

Isolation Forest Outlier Distribution:
Outlier_Flag
Inlier     11760
Outlier      240
Name: count, dtype: int64

Isolation Forest Outlier Percentage:
Outlier_Flag
Inlier     98.0
Outlier     2.0
Name: proportion, dtype: float64

99th Percentile Verification:
Age: Maximum = 69.00
Monthly_Income: Maximum = 535,803.00
Plot_Budget: Maximum = 32,938,671.26
Preferred_Plot_Size_SqFt: Maximum = 3,404.48
Distance_to_City_Center_km: Maximum = 47.20

Annual Income relationship: True


STEP 10 :- Final Validation & Export.

In [10]:
# Previous enquiry must imply site visit.

df.loc[
    df['Previous_Enquiry'] == 1,
    'Site_Visit'
] = 1


# Purchase must imply booking.

df.loc[
    df['Purchased'] == 1,
    'Booking_Done'
] = 1


# No purchase means Purchase_Value = 0.

df.loc[
    df['Purchased'] == 0,
    'Purchase_Value'
] = 0.0


# Every purchased plot must have a positive Purchase_Value.

buyer_invalid_value = (
    (df['Purchased'] == 1) &
    (df['Purchase_Value'] <= 0)
)

print(
    "Buyers with invalid Purchase_Value before correction:",
    buyer_invalid_value.sum()
)


# First use Plot_Budget where it is positive

use_budget = (
    buyer_invalid_value &
    (df['Plot_Budget'] > 0)
)

df.loc[
    use_budget,
    'Purchase_Value'
] = df.loc[
    use_budget,
    'Plot_Budget'
]


# If Plot_Budget is also invalid, use the median Purchase_Value of valid buyers.

remaining_invalid = (
    (df['Purchased'] == 1) &
    (df['Purchase_Value'] <= 0)
)

valid_buyer_median = df.loc[
    (df['Purchased'] == 1) &
    (df['Purchase_Value'] > 0),
    'Purchase_Value'
].median()

df.loc[
    remaining_invalid,
    'Purchase_Value'
] = valid_buyer_median


print(
    "Buyers with invalid Purchase_Value after correction:",
    (
        (df['Purchased'] == 1) &
        (df['Purchase_Value'] <= 0)
    ).sum()
)

# Recalculate Annual Income one final time

df['Annual_Income'] = (
    df['Monthly_Income'] * 12
)


# FINAL VALIDATION

print("\n")
print("FINAL DATA CHECK:")

print("Rows:", len(df))
print("Columns:", len(df.columns))


# Missing values

missing_values = df.isna().sum().sum()

print("\nMissing values:", missing_values)


# Duplicate rows

duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)


# Duplicate Customer IDs

duplicate_ids = df['Customer_ID'].duplicated().sum()

print("Duplicate Customer IDs:", duplicate_ids)


# Binary validation

print("\nBinary column validation:")

for col in binary_cols:

    invalid_values = ~df[col].isin([0, 1])

    print(
        f"{col}:",
        invalid_values.sum(),
        "invalid values"
    )


# BUSINESS LOGIC VALIDATION

enquiry_without_visit = (
    (df['Previous_Enquiry'] == 1) &
    (df['Site_Visit'] != 1)
).sum()

purchase_without_booking = (
    (df['Purchased'] == 1) &
    (df['Booking_Done'] != 1)
).sum()

non_purchase_with_value = (
    (df['Purchased'] == 0) &
    (df['Purchase_Value'] != 0)
).sum()

buyer_without_value = (
    (df['Purchased'] == 1) &
    (df['Purchase_Value'] <= 0)
).sum()


print("\nBusiness Logic Validation:")
print(
    "Enquiry without Site Visit:",
    enquiry_without_visit
)

print(
    "Purchase without Booking:",
    purchase_without_booking
)

print(
    "Non-purchase with Purchase Value:",
    non_purchase_with_value
)

print(
    "Purchased with zero/negative Purchase Value:",
    buyer_without_value
)


# FINANCIAL VALIDATION

annual_income_error = ~np.isclose(
    df['Annual_Income'],
    df['Monthly_Income'] * 12
)

print(
    "\nAnnual Income inconsistencies:",
    annual_income_error.sum()
)


# PURCHASE PROBABILITY VALIDATION

invalid_probability = (
    (df['Purchase_Probability'] < 0) |
    (df['Purchase_Probability'] > 1)
).sum()

print(
    "Purchase Probability outside 0-1:",
    invalid_probability
)


# OUTLIER SUMMARY

print("\nOutlier distribution:")
print(df['Outlier_Flag'].value_counts())


# FINAL STATUS

all_checks_passed = (
    missing_values == 0 and
    duplicate_rows == 0 and
    duplicate_ids == 0 and
    enquiry_without_visit == 0 and
    purchase_without_booking == 0 and
    non_purchase_with_value == 0 and
    buyer_without_value == 0 and
    annual_income_error.sum() == 0 and
    invalid_probability == 0
)

output_file = "real_estate_user_cleaned.csv"

df.to_csv(output_file, index = False)

Buyers with invalid Purchase_Value before correction: 2
Buyers with invalid Purchase_Value after correction: 0


FINAL DATA CHECK:
Rows: 12000
Columns: 27

Missing values: 0
Duplicate rows: 0
Duplicate Customer IDs: 0

Binary column validation:
Loan_Required: 0 invalid values
Previous_Enquiry: 0 invalid values
Site_Visit: 0 invalid values
Negotiation_Done: 0 invalid values
Booking_Done: 0 invalid values
Purchased: 0 invalid values

Business Logic Validation:
Enquiry without Site Visit: 0
Purchase without Booking: 0
Non-purchase with Purchase Value: 0
Purchased with zero/negative Purchase Value: 0

Annual Income inconsistencies: 0
Purchase Probability outside 0-1: 0

Outlier distribution:
Outlier_Flag
Inlier     11760
Outlier      240
Name: count, dtype: int64
